# ABSA Pipeline: Aspect-Based Sentiment Analysis
## Dataset: IMDb Reviews in Spanish (50,000 filas)

**Arquitectura:**
- **SpaCy** (`es_core_news_md`): Extracción sintáctica de pares (Aspecto, Adjetivo)
- **BERT** (`pysentimiento/robertuito-sentiment-analysis`): Clasificación de sentimiento por fragmento
- **Output**: Parquet estructurado para dashboard Streamlit

---

### Celda 1: Instalación e Importación de Librerías

**Objetivo:** Instalar de forma silenciosa todas las dependencias necesarias y descargar el modelo de SpaCy en español.

**Buenas prácticas aplicadas:**
- Uso de `-q` para suprimir logs innecesarios de pip
- Instalación de `pyarrow` para manejo eficiente de archivos Parquet
- Descarga del modelo `es_core_news_md` (balance entre precisión sintáctica y velocidad)

In [ ]:
# Celda 1: Instalación de Dependencias
# Ejecutar UNA SOLA VEZ al inicio de la sesión de Colab

!pip install -q pandas pyarrow spacy transformers torch tqdm kaggle

# Descargar modelo de SpaCy para español (medium = buen balance velocidad/precisión)
!python -m spacy download es_core_news_md -q

print("✅ Instalación completada. Reiniciar runtime si se solicita.")

In [ ]:
# Celda 1 (continuación): Importaciones

import os
import sys
import gc
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import torch
import spacy
from tqdm.notebook import tqdm

# Verificar disponibilidad de GPU
print(f"🔥 PyTorch versión: {torch.__version__}")
print(f"🚀 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Advertencia: No se detectó GPU. El procesamiento será más lento.")

### Celda 2: Configuración del Entorno

**Objetivo:** Montar Google Drive, descargar el dataset desde Kaggle y configurar rutas y logging.

**Estrategia de datos:**
- Montar Google Drive para persistir checkpoints y resultados
- Descargar dataset usando API de Kaggle (requiere `kaggle.json` en Drive)
- Cargar CSV con tipos optimizados (`category` para sentimiento)
- **Muestreo configurable**: `SAMPLE_SIZE` permite probar con subsets (ej: 1000) antes de procesar las 50k filas

**Manejo de excepciones:**
- Si Kaggle falla, intentar cargar desde Drive directamente
- Validar que las columnas requeridas existan (`review_es`, `sentiment`)

In [ ]:
# Celda 2: Montar Google Drive y Configurar Rutas

from google.colab import drive

# Montar Google Drive (requiere autorización)
drive.mount('/content/drive')

# Rutas del proyecto (ajustar según tu estructura en Drive)
BASE_PATH = Path("/content/drive/MyDrive/proyecto-absa-imdb")
DATA_RAW = BASE_PATH / "data" / "raw"
DATA_PROCESSED = BASE_PATH / "data" / "processed"
LOGS_PATH = BASE_PATH / "logs"

# Crear directorios si no existen
for path in [DATA_RAW, DATA_PROCESSED, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 {path}")

print("\n✅ Entorno configurado correctamente.")

In [ ]:
# Celda 2: Configuración de Logging

log_file = LOGS_PATH / f"absa_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)
logger.info("🚀 Pipeline ABSA iniciado")

In [ ]:
# Celda 2: Descarga del Dataset desde Kaggle

import zipfile

KAGGLE_DATASET = "lucamla/imdb-reviews-in-spanish"  # Ajustar si cambia el slug
CSV_FILENAME = "IMDB Dataset Spanish.csv"

csv_path = DATA_RAW / CSV_FILENAME

if not csv_path.exists():
    logger.info("📥 Descargando dataset desde Kaggle...")
    
    # Configurar API de Kaggle (asume kaggle.json en /content/drive/MyDrive/)
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json 2>/dev/null
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Descargar dataset
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_RAW}
    
    # Descomprimir si viene en zip
    zip_files = list(DATA_RAW.glob("*.zip"))
    if zip_files:
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall(DATA_RAW)
        logger.info(f"📦 Dataset descomprimido: {zip_files[0].name}")
else:
    logger.info("📂 Dataset ya existe en cache. Omitiendo descarga.")

In [ ]:
# Celda 2: Carga del Dataset con Tipos Optimizados

# Configuración de muestreo (ajustar según necesidad)
SAMPLE_SIZE = 5000  # 1000 para pruebas, 50000 para producción completa

logger.info(f"🔄 Cargando dataset con muestreo: {SAMPLE_SIZE} filas")

# Encontrar archivo CSV (puede estar en subcarpeta tras descompresión)
csv_files = list(DATA_RAW.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"❌ No se encontró CSV en {DATA_RAW}")

# Seleccionar el CSV más grande (probablemente el dataset principal)
csv_path = max(csv_files, key=lambda p: p.stat().st_size)
logger.info(f"📄 Archivo detectado: {csv_path.name}")

# Cargar con tipos optimizados para ahorrar memoria
dtypes = {
    'line number': 'int32',
    'review_en': 'string',
    'review_es': 'string',
    'sentiment': 'category',
    'sentimiento': 'category'
}

df = pd.read_csv(csv_path, dtype=dtypes, usecols=list(dtypes.keys()))

# Aplicar muestreo estratificado por sentimiento para balance
if SAMPLE_SIZE < len(df):
    df = df.groupby('sentiment', group_keys=False).apply(
        lambda x: x.sample(int(np.rint(SAMPLE_SIZE * len(x) / len(df))), random_state=42)
    ).sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle final
    logger.info(f"✂️ Muestreo estratificado aplicado: {len(df)} filas")

# Validar columnas requeridas
assert 'review_es' in df.columns, "❌ Falta columna 'review_es'"
assert 'sentiment' in df.columns, "❌ Falta columna 'sentiment'"

# Eliminar filas con texto vacío
df = df.dropna(subset=['review_es'])
df = df[df['review_es'].str.len() > 10]  # Filtrar textos muy cortos

logger.info(f"✅ Dataset cargado: {len(df)} filas | Memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
logger.info(f"📊 Distribución de sentimientos: {dict(df['sentiment'].value_counts())}")

df.head(3)

### Celda 3: Módulo de Extracción Sintáctica (SpaCy)

**Objetivo:** Extraer pares `(Aspecto, Adjetivo)` de cada reseña usando SpaCy con procesamiento por lotes.

**Optimizaciones aplicadas:**
- **`nlp.pipe(batch_size=256)`**: Procesamiento vectorizado, evita bucles `for doc in nlp(textos)`
- **Deshabilitar componentes**: `disable=['ner', 'textcat', 'entity_linker']` para ahorrar CPU
- **Lematización**: Normaliza aspectos (ej: 'actuaciones' → 'actuación') para evitar duplicados
- **Manejo de errores**: Try-except por reseña, loguear sin detener el pipeline

**Lógica de extracción:**
1. Para cada oración, identificar sustantivos (`NOUN`, `PROPN`)
2. Buscar hijos con dependencia `amod` (adjetivo modificador)
3. Extraer lema del sustantivo (aspecto) y forma del adjetivo
4. Guardar fragmento original para contexto del BERT

In [ ]:
# Celda 3: Configuración del Modelo SpaCy

import spacy
from typing import List, Dict

# Configuración del modelo
SPACY_MODEL = "es_core_news_md"
BATCH_SIZE_SPACY = 256

logger.info(f"🔧 Cargando modelo SpaCy: {SPACY_MODEL}")

nlp = spacy.load(
    SPACY_MODEL,
    disable=["ner", "textcat", "entity_linker"]  # Deshabilitar componentes no necesarios
)

# Añadir sentencizer si no está presente (para segmentar por oraciones)
if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

logger.info(f"✅ Modelo cargado. Componentes activos: {nlp.pipe_names}")

In [ ]:
# Celda 3: Funciones de Extracción de Aspectos

def extraer_aspectos_adjetivos(doc: spacy.tokens.Doc) -> List[Dict]:
    """
    Extrae pares (sustantivo aspecto, adjetivo modificador) de un Doc de SpaCy.
    
    Busca tokens NOUN/PROPN y sus dependientes 'amod' (adjetivos modificadores).
    
    Args:
        doc: Documento procesado por SpaCy.
    
    Returns:
        Lista de diccionarios con aspecto, adjetivo y fragmento original.
    """
    resultados = []
    
    for sent in doc.sents:
        for token in sent:
            # Identificar sustantivos como posibles aspectos
            if token.pos_ in ("NOUN", "PROPN"):
                # Buscar adjetivos que dependan directamente del sustantivo
                adjetivos = [
                    child for child in token.children
                    if child.dep_ == "amod" and child.pos_ == "ADJ"
                ]
                
                for adj in adjetivos:
                    # Extraer fragmento original para contexto
                    inicio = min(token.i, adj.i)
                    fin = max(token.i, adj.i) + 1
                    fragmento = doc[inicio:fin].text
                    
                    resultados.append({
                        "aspecto_lematizado": token.lemma_.lower(),
                        "adjetivo": adj.text.lower(),
                        "adjetivo_lematizado": adj.lemma_.lower(),
                        "fragmento": fragmento,
                        "dep": adj.dep_,
                        "pos_aspecto": token.pos_,
                        "pos_adjetivo": adj.pos_
                    })
    
    return resultados

def procesar_lote_spacy(
    textos: List[str],
    ids: List[int],
    nlp: spacy.Language,
    batch_size: int = 256
) -> List[Dict]:
    """
    Procesa un lote de textos usando nlp.pipe() y extrae aspectos.
    
    Args:
        textos: Lista de reseñas en español.
        ids: Lista de IDs de reseñas.
        nlp: Modelo SpaCy cargado.
        batch_size: Tamaño de lote para nlp.pipe().
    
    Returns:
        Lista de diccionarios con metadata completa por aspecto.
    """
    resultados = []
    total_errores = 0
    total_aspectos = 0
    
    logger.info(f"📝 Procesando {len(textos)} textos con batch_size={batch_size}")
    
    for review_id, doc in tqdm(
        zip(ids, nlp.pipe(textos, batch_size=batch_size)),
        total=len(textos),
        desc="Extracción SpaCy"
    ):
        try:
            if not doc or len(doc.text.strip()) == 0:
                continue
            
            aspectos = extraer_aspectos_adjetivos(doc)
            total_aspectos += len(aspectos)
            
            for asp in aspectos:
                resultados.append({
                    "review_id": review_id,
                    "aspecto": asp["aspecto_lematizado"],
                    "adjetivo": asp["adjetivo"],
                    "adjetivo_lematizado": asp["adjetivo_lematizado"],
                    "fragmento": asp["fragmento"],
                    "texto_completo": doc.text
                })
        
        except Exception as e:
            total_errores += 1
            logger.warning(f"⚠️ Error en review_id={review_id}: {str(e)}")
            continue
    
    logger.info(
        f"✅ Extracción completada: {total_aspectos} aspectos | "
        f"{total_errores} errores"
    )
    
    return resultados

print("✅ Funciones de extracción definidas correctamente.")

In [ ]:
# Celda 3: Ejecución de la Extracción de Aspectos

# Liberar memoria antes de procesar
gc.collect()

# Preparar datos
textos = df['review_es'].tolist()
ids = df['line number'].tolist()

logger.info(f"🎯 Iniciando extracción de aspectos para {len(textos)} reseñas...")

# Ejecutar extracción (con barra de progreso integrada)
aspectos_extraidos = procesar_lote_spacy(
    textos=textos,
    ids=ids,
    nlp=nlp,
    batch_size=BATCH_SIZE_SPACY
)

# Convertir a DataFrame
df_aspectos = pd.DataFrame(aspectos_extraidos)

logger.info(f"📊 Total aspectos extraídos: {len(df_aspectos)}")
logger.info(f"📁 Memoria del DataFrame: {df_aspectos.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Mostrar resumen
print("\n🎭 Top 10 Aspectos más frecuentes:")
print(df_aspectos['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_aspectos['adjetivo'].value_counts().head(10))

# Mostrar ejemplos
print("\n💡 Ejemplos de extracción:")
df_aspectos[['review_id', 'aspecto', 'adjetivo', 'fragmento']].head(10)

In [ ]:
# Celda 3: Checkpoint Intermedio (Persistencia segura)

# Guardar progreso de SpaCy para poder reanudar si Colab se desconecta
checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"

df_aspectos.to_parquet(checkpoint_spacy, index=False, compression='snappy')

logger.info(f"💾 Checkpoint guardado: {checkpoint_spacy}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_spacy.stat().st_size / 1e6:.2f} MB")

# Liberar memoria de variables pesadas
del textos
del ids
del aspectos_extraidos
gc.collect()

print("✅ Checkpoint guardado. Listo para Celda 4 (BERT).")

### Celda 4: Módulo de Inferencia de Sentimiento (BERT)

**Objetivo:** Clasificar el sentimiento de cada fragmento extraído usando `pysentimiento/robertuito-sentiment-analysis` en GPU.

**Optimizaciones aplicadas:**
- **`pipeline(device=0)`**: Fuerza uso de GPU (CUDA)
- **`batch_size=64` o `128`**: Procesamiento por lotes para saturar la GPU
- **`truncation=True, max_length=128`**: Evita exceder límite de tokens del modelo
- **`torch.cuda.empty_cache()`**: Libera VRAM entre lotes grandes para evitar OOM
- **Manejo de errores**: Si un batch falla, se salta y continúa (no se pierden los anteriores)

**Mapeo de labels:**
- `POS` → positivo
- `NEG` → negativo
- `NEU` → neutral

In [ ]:
# Celda 4: Configuración del Pipeline BERT

from transformers import pipeline

# Configuración del modelo BERT para sentimiento en español
BERT_MODEL = "pysentimiento/robertuito-sentiment-analysis"
BATCH_SIZE_BERT = 64  # Ajustar según VRAM disponible (64 para T4, 128 para A100)
MAX_LENGTH = 128

# Verificar GPU antes de cargar modelo
if torch.cuda.is_available():
    device = 0
    print(f"🚀 Usando GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = -1
    print("⚠️ GPU no disponible. Usando CPU (será más lento).")

logger.info(f"🔧 Cargando modelo BERT: {BERT_MODEL}")

# Configurar pipeline de Hugging Face con truncamiento
classifier = pipeline(
    "sentiment-analysis",
    model=BERT_MODEL,
    tokenizer=BERT_MODEL,
    device=device,
    truncation=True,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE_BERT
)

print("✅ Pipeline BERT configurado correctamente.")

In [ ]:
# Celda 4: Inferencia por Lotes con BERT

def analizar_sentimiento_bert(
    fragmentos: List[str],
    classifier,
    batch_size: int = 64
) -> List[Dict]:
    """
    Clasifica el sentimiento de una lista de fragmentos usando BERT/RoBERTa.
    
    Args:
        fragmentos: Lista de strings (fragmentos extraídos por SpaCy).
        classifier: Pipeline de Hugging Face configurado.
        batch_size: Tamaño de lote para enviar a la GPU.
    
    Returns:
        Lista de diccionarios con sentimiento, label y score de confianza.
    """
    resultados = []
    total_fragmentos = len(fragmentos)
    errores = 0
    
    logger.info(f"🧠 Iniciando inferencia BERT: {total_fragmentos} fragmentos, batch={batch_size}")
    
    for i in tqdm(range(0, total_fragmentos, batch_size), desc="Inferencia BERT"):
        lote = fragmentos[i:i + batch_size]
        
        try:
            predicciones = classifier(lote)
            
            for pred in predicciones:
                resultados.append({
                    "sentimiento_bert": pred["label"],
                    "confianza": round(pred["score"], 4),
                    "label_original": pred["label"]
                })
        
        except Exception as e:
            errores += 1
            logger.error(f"❌ Error en batch {i//batch_size}: {str(e)}")
            # Agregar placeholders para mantener alineación
            for _ in lote:
                resultados.append({
                    "sentimiento_bert": "ERROR",
                    "confianza": 0.0,
                    "label_original": "ERROR"
                })
        
        # Liberar memoria GPU periódicamente (cada 10 batches)
        if i % (batch_size * 10) == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    logger.info(f"✅ Inferencia BERT completada: {len(resultados)} resultados | {errores} errores")
    return resultados

print("✅ Función de inferencia BERT definida.")

In [ ]:
# Celda 4: Ejecución de Inferencia BERT

# Cargar checkpoint de SpaCy si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
    if checkpoint_spacy.exists():
        df_aspectos = pd.read_parquet(checkpoint_spacy)
        logger.info(f"📂 Checkpoint SpaCy cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint de SpaCy. Ejecutar Celda 3 primero.")

# Preparar fragmentos para BERT
fragmentos = df_aspectos['fragmento'].tolist()

logger.info(f"🎯 Enviando {len(fragmentos)} fragmentos al modelo BERT...")

# Ejecutar inferencia
resultados_sentimiento = analizar_sentimiento_bert(
    fragmentos=fragmentos,
    classifier=classifier,
    batch_size=BATCH_SIZE_BERT
)

# Mapear sentimientos a etiquetas legibles
sentiment_map = {
    "POS": "positivo",
    "NEG": "negativo",
    "NEU": "neutral"
}

# Crear DataFrame de resultados BERT
df_sentimientos = pd.DataFrame(resultados_sentimiento)
df_sentimientos['sentimiento'] = df_sentimientos['sentimiento_bert'].map(sentiment_map).fillna('desconocido')

# Unir con aspectos
df_aspectos['sentimiento_bert'] = df_sentimientos['sentimiento']
df_aspectos['confianza'] = df_sentimientos['confianza']
df_aspectos['label_original'] = df_sentimientos['label_original']

# Mostrar distribución de sentimientos
print("\n📊 Distribución de Sentimientos por Aspecto:")
print(df_aspectos['sentimiento_bert'].value_counts())

print("\n💡 Ejemplos con sentimiento:")
print(df_aspectos[['aspecto', 'adjetivo', 'fragmento', 'sentimiento_bert', 'confianza']].head(10))

# Liberar memoria
del fragmentos
del resultados_sentimiento
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Celda 4 completada. Memoria liberada.")

In [ ]:
# Celda 4: Checkpoint BERT (Persistencia)

# Guardar progreso después de BERT
checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"

df_aspectos.to_parquet(checkpoint_bert, index=False, compression='snappy')

logger.info(f"💾 Checkpoint BERT guardado: {checkpoint_bert}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_bert.stat().st_size / 1e6:.2f} MB")

print("✅ Checkpoint BERT guardado. Listo para Celda 5 (Consolidación Final).")

### Celda 5: Orquestación Final y Guardado

**Objetivo:** Consolidar todos los resultados, calcular métricas agregadas por aspecto y exportar el archivo final optimizado para Streamlit.

**Procesos realizados:**
1. **Merge con datos originales**: Unir `sentiment` original del dataset con predicciones BERT
2. **Agregación por aspecto**: Calcular métricas desglosadas (conteo, confianza promedio, sentimiento dominante)
3. **Normalización**: Eliminar aspectos poco frecuentes (ruido) y estandarizar formato
4. **Exportación final**: Parquet comprimido con todas las columnas necesarias para el dashboard

**Estructura del output final (`aspectos_sentimientos_final.parquet`):**
- `review_id`: ID original de la reseña
- `aspecto`: Sustantivo lematizado (aspecto extraído)
- `adjetivo`: Adjetivo tal cual aparece en el texto
- `fragmento`: Contexto completo del par aspecto-adjetivo
- `sentimiento_bert`: Etiqueta predicha (positivo/negativo/neutral)
- `confianza`: Score de confianza del modelo BERT
- `sentiment_original`: Polaridad original del dataset (pos/neg)
- `metricas_aspecto`: Conteo, confianza promedio y sentimiento dominante por aspecto

In [ ]:
# Celda 5: Consolidación Final y Exportación

# Cargar checkpoint si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"
    if checkpoint_bert.exists():
        df_aspectos = pd.read_parquet(checkpoint_bert)
        logger.info(f"📂 Checkpoint BERT cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint BERT. Ejecutar Celda 4 primero.")

# ───────────────────────────────────────────────
# PASO 1: Merge con datos originales del dataset
# ───────────────────────────────────────────────

logger.info("🔄 Iniciando consolidación final...")

# Preparar dataset original con columnas necesarias
if 'df' in globals():
    df_original = df[['line number', 'sentiment', 'sentimiento']].copy()
    df_original.columns = ['review_id', 'sentiment_original', 'sentimiento_original']
    
    # Merge: cada aspecto hereda el sentiment original de su review
    df_final = df_aspectos.merge(
        df_original,
        on='review_id',
        how='left'
    )
else:
    logger.warning("⚠️ DataFrame original no disponible. Continuando sin merge.")
    df_final = df_aspectos.copy()
    df_final['sentiment_original'] = 'unknown'
    df_final['sentimiento_original'] = 'unknown'

# ───────────────────────────────────────────────
# PASO 2: Agregación de métricas por aspecto
# ───────────────────────────────────────────────

logger.info("📊 Calculando métricas agregadas por aspecto...")

# Calcular métricas desglosadas por aspecto
metricas_aspecto = df_final.groupby('aspecto').agg(
    total_apariciones=('aspecto', 'size'),
    confianza_promedio=('confianza', 'mean'),
    sentimiento_dominante=('sentimiento_bert', lambda x: x.mode().iloc[0] if not x.mode().empty else 'neutral'),
    pct_positivo=('sentimiento_bert', lambda x: (x == 'positivo').mean() * 100),
    pct_negativo=('sentimiento_bert', lambda x: (x == 'negativo').mean() * 100),
    pct_neutral=('sentimiento_bert', lambda x: (x == 'neutral').mean() * 100)
).reset_index()

# Filtrar aspectos con poca frecuencia (ruido): mínimo 5 apariciones
MIN_FRECUENCIA = 5
aspectos_validos = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA]['aspecto'].tolist()

df_final = df_final[df_final['aspecto'].isin(aspectos_validos)].copy()
metricas_aspecto = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA].copy()

logger.info(f"🎯 Aspectos filtrados (>= {MIN_FRECUENCIA} apariciones): {len(aspectos_validos)}")

# ───────────────────────────────────────────────
# PASO 3: Optimizar tipos de datos para exportación
# ───────────────────────────────────────────────

# Convertir a tipos eficientes
df_final['sentimiento_bert'] = df_final['sentimiento_bert'].astype('category')
df_final['sentiment_original'] = df_final['sentiment_original'].astype('category')
df_final['aspecto'] = df_final['aspecto'].astype('category')
df_final['confianza'] = df_final['confianza'].astype('float32')

# ───────────────────────────────────────────────
# PASO 4: Exportar resultado final a Parquet
# ───────────────────────────────────────────────

OUTPUT_FILE = DATA_PROCESSED / "aspectos_sentimientos_final.parquet"
METRICAS_FILE = DATA_PROCESSED / "metricas_por_aspecto.parquet"

# Guardar DataFrame principal (aspecto por fila)
df_final.to_parquet(OUTPUT_FILE, index=False, compression='snappy')
logger.info(f"💾 Archivo final guardado: {OUTPUT_FILE}")
logger.info(f"📦 Tamaño: {OUTPUT_FILE.stat().st_size / 1e6:.2f} MB")

# Guardar métricas agregadas por aspecto
metricas_aspecto.to_parquet(METRICAS_FILE, index=False, compression='snappy')
logger.info(f"💾 Métricas por aspecto guardadas: {METRICAS_FILE}")

# ───────────────────────────────────────────────
# RESUMEN FINAL
# ───────────────────────────────────────────────

print("\n" + "="*60)
print("🎉 PIPELINE ABSA COMPLETADO CON ÉXITO")
print("="*60)
print(f"📊 Total de reseñas procesadas: {df_final['review_id'].nunique()}")
print(f"🎭 Total de pares aspecto-adjetivo: {len(df_final)}")
print(f"🏷️  Aspectos únicos encontrados: {df_final['aspecto'].nunique()}")
print(f"📁 Archivo final: {OUTPUT_FILE.name}")
print(f"📈 Métricas por aspecto: {METRICAS_FILE.name}")
print("="*60)

print("\n📊 Distribución de Sentimientos (BERT):")
print(df_final['sentimiento_bert'].value_counts())

print("\n🎭 Top 10 Aspectos más mencionados:")
print(df_final['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_final['adjetivo'].value_counts().head(10))

print("\n💡 Vista previa del output final:")
print(df_final[['review_id', 'aspecto', 'adjetivo', 'sentimiento_bert', 'confianza']].head())

# ───────────────────────────────────────────────
# LIMPIEZA FINAL DE MEMORIA
# ───────────────────────────────────────────────

logger.info("🧹 Limpiando memoria...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Pipeline finalizado. Todo listo para Streamlit! 🚀")

### 🎉 Pipeline Completado

**Archivos generados en `data/processed/`:**
1. `aspectos_sentimientos_final.parquet` — Dataset completo con aspectos y sentimientos (listo para Streamlit)
2. `metricas_por_aspecto.parquet` — Métricas agregadas por aspecto
3. `checkpoint_aspectos_spacy.parquet` — Checkpoint intermedio de SpaCy
4. `checkpoint_aspectos_sentimientos.parquet` — Checkpoint intermedio de BERT

**Próximos pasos:**
- Descargar los archivos Parquet desde Google Drive
- Construir dashboard de Streamlit consumiendo `aspectos_sentimientos_final.parquet`
- Visualizar: distribución de aspectos, nubes de adjetivos, evolución de sentimiento por review
